# Head Direction Cells in the Anterior Thalamus and Postsubiculum

This notebook demonstrates head direction (HD) tuning in extracellular recordings from
freely moving mice, using data from the DANDI Archive.

**Dataset:** [DANDI:000056](https://dandiarchive.org/dandiset/000056) — "Internally
organized mechanisms of the head direction sense" (Peyrache lab). Mice were implanted
with silicon probes in the anterior thalamic nucleus (ADn) and/or postsubiculum (PoSub)
and recorded while freely foraging in an open arena. Two head-mounted LEDs (red and blue)
were tracked to reconstruct the animal's instantaneous head direction.

**Session used:** `sub-Mouse24/sub-Mouse24_ses-Mouse24-131213` (the smallest session in
the dandiset, ~1.3 GB, streamed directly from S3 rather than downloaded in full).

**Analysis:**
1. Load spikes, LED positions, and behavioral-state epochs via Pynapple.
2. Compute the instantaneous head-direction angle from the vector between the two LEDs.
3. Restrict to waking behavior and compute circular (angular) tuning curves per neuron.
4. Identify head-direction cells using the Rayleigh vector length, with a shuffle test
   for significance.
5. Visualize tuning curves, sort neurons by preferred direction, and Bayesian-decode
   head direction from the population to show that HD cells collectively track heading.

In [1]:
import os

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pynapple as nap
import remfile
from pynwb import NWBHDF5IO
from tqdm import tqdm

np.random.seed(0)

FIGDIR = "figures"
os.makedirs(FIGDIR, exist_ok=True)

## Load the NWB File (Streaming from DANDI)

The file is streamed directly from the DANDI S3 bucket with `remfile`, using a local
disk cache so repeated reads of the same byte ranges don't re-download data.

In [2]:
S3_URL = "https://dandiarchive.s3.amazonaws.com/blobs/f00/e5c/f00e5c3a-9435-42df-aace-9b6952563479"

disk_cache = remfile.DiskCache("/tmp/remfile_cache")
rem_file = remfile.File(S3_URL, disk_cache=disk_cache)
h5_file = h5py.File(rem_file, "r")
io = NWBHDF5IO(file=h5_file, load_namespaces=True)
nwbfile = io.read()
nwb = nap.NWBFile(nwbfile)
print(nwb)

Mouse24-131213
┍━━━━━━━━━━━━━━━━━━━━━━━━━┯━━━━━━━━━━━━━┑
│ Keys                    │ Type        │
┝━━━━━━━━━━━━━━━━━━━━━━━━━┿━━━━━━━━━━━━━┥
│ units                   │ TsGroup     │
│ LFP                     │ TsdFrame    │
│ states                  │ IntervalSet │
│ SubjectPosition/RedLED  │ TsdFrame    │
│ SubjectPosition/BlueLED │ TsdFrame    │
│ RedLED                  │ TsdFrame    │
│ BlueLED                 │ TsdFrame    │
┕━━━━━━━━━━━━━━━━━━━━━━━━━┷━━━━━━━━━━━━━┙


In [3]:
units = nwb["units"]
red_led = nwb["RedLED"]
blue_led = nwb["BlueLED"]
states = nwb["states"]

print(f"Number of units (raw): {len(units)}")
print(f"Firing rate range: {units.get_info('rate').min():.2f} - {units.get_info('rate').max():.2f} Hz")
print(states)

Number of units (raw): 22
Firing rate range: 0.00 - 99.85 Hz
index    start    end      label
0        1.0      2484.0   Awake
1        2485.0   3018.0   Non-REM
2        3019.0   3033.0   Awake
3        3034.0   3397.0   Non-REM
4        3398.0   3580.0   REM
5        3581.0   3601.0   Awake
6        3602.0   4546.0   Non-REM
...      ...      ...      ...
30       7332.0   9040.0   Awake
31       9041.0   9908.0   Non-REM
32       9909.0   10041.0  REM
33       10042.0  10087.0  Awake
34       10088.0  10690.0  Non-REM
35       10691.0  10745.0  REM
36       10746.0  10753.0  Awake
shape: (37, 2), time unit: sec.


/Users/bdichter/miniconda3/lib/python3.12/site-packages/pynapple/core/time_index.py:109: UserWarning: timestamps are not sorted
  warn("timestamps are not sorted", UserWarning)


In [4]:
# drop silent units (zero spikes for the whole session) - these break circular
# statistics downstream (0/0 resultant vector length) and can't be meaningfully
# tested for tuning anyway
silent = units.get_info("rate") == 0
print(f"Dropping {silent.sum()} silent unit(s): {list(units.index[silent])}")
units = units[~silent]
print(f"Number of units (analyzed): {len(units)}")

Dropping 1 silent unit(s): [np.int64(13)]
Number of units (analyzed): 21


## Compute Head Direction from LED Positions

The head direction is the angle of the vector pointing from the blue LED to the red
LED, in the plane of the arena floor. Frames where either LED was not detected are
coded as `(-1, -1)` in the raw data and must be dropped before computing the angle.

In [5]:
red_xy = red_led.values
blue_xy = blue_led.values
valid = (red_xy[:, 0] != -1) & (blue_xy[:, 0] != -1)
print(f"Valid tracking frames: {valid.sum()} / {len(valid)} ({100 * valid.mean():.1f}%)")

dx = red_xy[:, 0] - blue_xy[:, 0]
dy = red_xy[:, 1] - blue_xy[:, 1]
angle = np.mod(np.arctan2(dy, dx), 2 * np.pi)

head_direction = nap.Tsd(t=red_led.index.values[valid], d=angle[valid])
print(head_direction)

Valid tracking frames: 411832 / 420099 (98.0%)
Time (s)
----------  -------
0.0512      5.31793
0.0768      5.31793
0.1024      5.31219
0.128       5.26683
0.1536      5.22053
0.1792      5.25042
0.2048      5.28373
...
10754.3552  6.02258
10754.3808  6.02258
10754.4064  6.02258
10754.432   6.02258
10754.4576  6.00647
10754.4832  5.96384
10754.5088  6.01265
dtype: float64, shape: (411832,)


## Restrict to Waking Behavior

The `states` epochs distinguish Awake, Non-REM sleep, and REM sleep. Head direction
tuning during active foraging is computed only during Awake epochs.

In [6]:
wake_ep = states[states.label == "Awake"]
print(wake_ep)

head_direction_wake = head_direction.restrict(wake_ep)
print(f"Wake head-direction samples: {len(head_direction_wake)}")

index    start    end      label
0        1.0      2484.0   Awake
1        3019.0   3033.0   Awake
2        3581.0   3601.0   Awake
3        4670.0   4692.0   Awake
4        4903.0   4917.0   Awake
5        5007.0   5068.0   Awake
6        5564.0   5580.0   Awake
...      ...      ...      ...
8        5989.0   6015.0   Awake
9        6225.0   6244.0   Awake
10       6686.0   6840.0   Awake
11       7284.0   7309.0   Awake
12       7332.0   9040.0   Awake
13       10042.0  10087.0  Awake
14       10746.0  10753.0  Awake
shape: (15, 2), time unit: sec.
Wake head-direction samples: 172648


## Raw Data Overview

Before computing tuning curves, visualize the raw spike rasters, the LED trajectory,
and the head-direction time series to confirm the data look sensible.

In [7]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=False)

# Spike raster (first 60s of an awake epoch)
example_ep = nap.IntervalSet(start=wake_ep.start[0], end=wake_ep.start[0] + 60)
spikes_ex = units.restrict(example_ep)
for i, (idx, ts) in enumerate(spikes_ex.items()):
    axes[0].vlines(ts.index.values, i, i + 0.8, color="k", linewidth=0.5)
axes[0].set_ylabel("Unit #")
axes[0].set_title(f"Spike raster, 60 s awake example (starting t={wake_ep.start[0]:.0f} s)")
axes[0].set_xlabel("Time (s)")

# LED trajectory
red_ex = red_led.restrict(example_ep).values
blue_ex = blue_led.restrict(example_ep).values
axes[1].plot(red_xy[valid, 0], red_xy[valid, 1], ",", color="lightgray", alpha=0.3, label="full session (red LED)")
axes[1].plot(red_ex[:, 0], red_ex[:, 1], "-o", color="red", markersize=2, linewidth=0.5, label="60s example (red LED)")
axes[1].plot(blue_ex[:, 0], blue_ex[:, 1], "-o", color="blue", markersize=2, linewidth=0.5, label="60s example (blue LED)")
axes[1].set_xlabel("x (px)")
axes[1].set_ylabel("y (px)")
axes[1].set_title("Head LED tracking in the arena")
axes[1].legend(loc="upper right", fontsize=8)
axes[1].set_aspect("equal")

# Head direction time series
hd_ex = head_direction.restrict(example_ep)
axes[2].plot(hd_ex.index.values, np.degrees(hd_ex.values), ".", markersize=2)
axes[2].set_xlabel("Time (s)")
axes[2].set_ylabel("Head direction (deg)")
axes[2].set_title("Head direction over the same 60 s example")

plt.tight_layout()
plt.savefig(f"{FIGDIR}/01_raw_data_overview.png", dpi=150)
plt.close()

## Compute Angular Tuning Curves

For each unit, the firing rate is binned as a function of head direction (60 bins
spanning 0-360 deg), restricted to waking epochs.

In [8]:
N_BINS = 60
tuning_curves = nap.compute_tuning_curves(
    units, head_direction_wake, bins=N_BINS, range=[(0, 2 * np.pi)], epochs=wake_ep
)
angles = tuning_curves["0"].values
print(tuning_curves)

<xarray.DataArray (unit: 21, 0: 60)> Size: 10kB
array([[ 3.92685789,  2.99693504,  2.48698809, ...,  2.71966594,
         3.59953836,  4.57104463],
       [70.56125446, 64.9798303 , 65.69449273, ..., 66.42690917,
        70.19958878, 70.08554172],
       [24.3701138 , 20.94569217, 21.70950568, ..., 21.22177686,
        23.88237861, 23.85513914],
       ...,
       [ 0.38762975,  0.4489927 ,  0.55771328, ...,  0.78702662,
         0.65289956,  0.62851864],
       [ 0.51824412,  0.35773402,  0.33875918, ...,  0.98261903,
         0.82471523,  0.75422236],
       [ 0.67413869,  0.62785972,  0.56184449, ...,  0.6286899 ,
         0.80323827,  0.59994961]])
Coordinates:
  * unit     (unit) int64 168B 0 1 2 3 4 5 6 7 8 ... 12 14 15 16 17 18 19 20 21
  * 0        (0) float64 480B 0.05236 0.1571 0.2618 0.3665 ... 6.021 6.126 6.231
Attributes:
    occupancy:  [ 8850. 10215.  9026.  9818.  6284.  5388.  3984.  3822.  289...
    bin_edges:  [array([0.        , 0.10471976, 0.20943951, 0.31415927, 

## Identify Head-Direction Cells

A neuron's directional selectivity is summarized by the mean resultant (Rayleigh)
vector length of its tuning curve, `R`, which ranges from 0 (uniform firing across
directions) to 1 (firing concentrated at a single direction). Significance is assessed
with a shuffle test: head-direction timestamps are randomly time-shifted (circularly,
within the wake epoch) 500 times, tuning curves are recomputed, and the true `R` is
compared to the shuffle distribution.

In [9]:
def resultant_vector_length(rates, angles):
    """Mean resultant vector length of a circular tuning curve."""
    weights = rates / rates.sum()
    return np.abs(np.sum(weights * np.exp(1j * angles)))


unit_ids = tuning_curves["unit"].values
true_R = np.array(
    [resultant_vector_length(tuning_curves.sel(unit=u).values, angles) for u in unit_ids]
)

In [10]:
N_SHUFFLES = 500
n_samples = len(head_direction_wake)
shuffle_R = np.zeros((N_SHUFFLES, len(unit_ids)))

# circular shuffle: roll the HD values relative to their own timestamps, so
# each shuffle keeps the HD trace's autocorrelation intact but destroys its
# alignment with the spike trains
for i in tqdm(range(N_SHUFFLES), desc="Shuffling head direction"):
    shift_samples = np.random.randint(int(0.1 * n_samples), int(0.9 * n_samples))
    shuffled_values = np.roll(head_direction_wake.values, shift_samples)
    shuffled_hd = nap.Tsd(t=head_direction_wake.index.values, d=shuffled_values)
    shuf_tc = nap.compute_tuning_curves(
        units, shuffled_hd, bins=N_BINS, range=[(0, 2 * np.pi)], epochs=wake_ep
    )
    shuffle_R[i] = np.array(
        [resultant_vector_length(shuf_tc.sel(unit=u).values, angles) for u in unit_ids]
    )

p_values = np.mean(shuffle_R >= true_R[None, :], axis=0)
is_hd_cell = p_values < 0.01

print(f"{is_hd_cell.sum()} / {len(unit_ids)} units classified as head-direction cells (p < 0.01)")
for u, r, p, hd in zip(unit_ids, true_R, p_values, is_hd_cell):
    print(f"  unit {u}: R={r:.3f}, p={p:.3f}, HD cell={hd}")

Shuffling head direction:   0%|          | 0/500 [00:00<?, ?it/s]

Shuffling head direction:   0%|          | 1/500 [00:00<00:54,  9.10it/s]

Shuffling head direction:   0%|          | 2/500 [00:00<00:54,  9.08it/s]

Shuffling head direction:   1%|          | 3/500 [00:00<00:55,  9.01it/s]

Shuffling head direction:   1%|          | 4/500 [00:00<00:54,  9.13it/s]

...

Shuffling head direction: 100%|█████████▉| 498/500 [00:55<00:00,  9.04it/s]

Shuffling head direction: 100%|█████████▉| 499/500 [00:56<00:00,  8.92it/s]

Shuffling head direction: 100%|██████████| 500/500 [00:56<00:00,  8.92it/s]

Shuffling head direction: 100%|██████████| 500/500 [00:56<00:00,  8.90it/s]

9 / 21 units classified as head-direction cells (p < 0.01)
  unit 0: R=0.327, p=0.766, HD cell=False
  unit 1: R=0.050, p=0.446, HD cell=False
  unit 2: R=0.036, p=0.334, HD cell=False
  unit 3: R=0.538, p=0.000, HD cell=True
  unit 4: R=0.085, p=0.512, HD cell=False
  unit 5: R=0.099, p=0.016, HD cell=False
  unit 6: R=0.774, p=0.000, HD cell=True
  unit 7: R=0.039, p=0.032, HD cell=False
  unit 8: R=0.156, p=0.000, HD cell=True
  unit 9: R=0.108, p=0.000, HD cell=True
  unit 10: R=0.172, p=0.000, HD cell=True
  unit 11: R=0.029, p=0.472, HD cell=False
  unit 12: R=0.026, p=0.832, HD cell=False
  unit 14: R=0.179, p=0.018, HD cell=False
  unit 15: R=0.077, p=0.526, HD cell=False
  unit 16: R=0.072, p=0.000, HD cell=True
  unit 17: R=0.162, p=0.000, HD cell=True
  unit 18: R=0.831, p=0.000, HD cell=True
  unit 19: R=0.221, p=0.000, HD cell=True
  unit 20: R=0.156, p=0.012, HD cell=False
  unit 21: R=0.221, p=0.356, HD cell=False


## Visualize Directional Tuning

Polar plots of firing rate vs. head direction for the significant HD cells, and a
summary of resultant vector length vs. shuffle-test p-value for the full population.

In [11]:
hd_unit_ids = unit_ids[is_hd_cell]
n_hd = len(hd_unit_ids)
n_cols = 4
n_rows = int(np.ceil(n_hd / n_cols))

fig = plt.figure(figsize=(3.2 * n_cols, 3.2 * n_rows))
for i, u in enumerate(hd_unit_ids):
    ax = fig.add_subplot(n_rows, n_cols, i + 1, projection="polar")
    rates = tuning_curves.sel(unit=u).values
    # close the loop for a continuous polar line
    theta = np.append(angles, angles[0])
    r = np.append(rates, rates[0])
    ax.plot(theta, r, color="C0")
    ax.fill(theta, r, color="C0", alpha=0.2)
    ax.set_title(f"unit {u}\nR={true_R[unit_ids == u][0]:.2f}", fontsize=10, y=1.2)
    ax.set_xticks(np.radians([0, 45, 90, 135, 180, 225, 270, 315]))
    ax.set_xticklabels(["0", "", "90", "", "180", "", "270", ""])
    ax.set_yticklabels([])

plt.tight_layout()
plt.savefig(f"{FIGDIR}/02_polar_tuning_curves.png", dpi=150)
plt.close()

In [12]:
fig, ax = plt.subplots(figsize=(6, 5))
colors = np.where(is_hd_cell, "C3", "gray")
ax.scatter(true_R, p_values, c=colors, s=40, edgecolor="k", linewidth=0.5)
ax.axhline(0.01, color="k", linestyle="--", linewidth=1, label="p = 0.01")
ax.set_xlabel("Resultant vector length (R)")
ax.set_ylabel("Shuffle-test p-value")
ax.set_title("Head-direction tuning strength vs. significance")
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIGDIR}/03_R_vs_pvalue.png", dpi=150)
plt.close()

## Population Tuning Curve Heatmap

Sorting all units by preferred direction reveals that the significant HD cells tile
the full range of head directions, a hallmark of an HD-cell population that can
collectively represent any heading.

In [13]:
preferred_direction = angles[np.argmax(tuning_curves.values, axis=1)]
sort_order = np.argsort(preferred_direction)

normalized_tc = tuning_curves.values / tuning_curves.values.max(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(
    normalized_tc[sort_order],
    aspect="auto",
    extent=[0, 360, len(unit_ids), 0],
    cmap="viridis",
)
for rank, orig_idx in enumerate(sort_order):
    if is_hd_cell[orig_idx]:
        ax.text(365, rank + 0.7, "*", color="red", fontsize=12)
ax.set_xlabel("Head direction (deg)")
ax.set_ylabel("Unit (sorted by preferred direction)")
ax.set_title("Normalized firing rate vs. head direction, all units\n(* = significant HD cell)")
plt.colorbar(im, ax=ax, label="Normalized rate")
plt.tight_layout()
plt.savefig(f"{FIGDIR}/04_population_heatmap.png", dpi=150)
plt.close()

## Bayesian Decoding of Head Direction from Population Activity

If the significant HD cells genuinely encode heading, their joint spiking activity
should allow head direction to be reconstructed on held-out data. Tuning curves are
fit on the first half of the wake epochs and used to Bayesian-decode head direction
on the second half.

In [14]:
hd_units = units[hd_unit_ids]

mid_time = wake_ep.start[0] + wake_ep.tot_length() / 2
train_ep = wake_ep.intersect(nap.IntervalSet(start=wake_ep.start[0], end=mid_time))
test_ep = wake_ep.intersect(nap.IntervalSet(start=mid_time, end=wake_ep.end[-1]))

train_hd = head_direction.restrict(train_ep)
train_tc = nap.compute_tuning_curves(
    hd_units, train_hd, bins=N_BINS, range=[(0, 2 * np.pi)], epochs=train_ep
)

BIN_SIZE = 0.3  # seconds
decoded_hd, decoded_proba = nap.decode_bayes(
    train_tc, hd_units, test_ep, bin_size=BIN_SIZE, sliding_window_size=3
)

true_hd_test = head_direction.restrict(test_ep)

In [15]:
# circular decoding error, in degrees
true_hd_interp = true_hd_test.interpolate(decoded_hd)
error = np.degrees(np.angle(np.exp(1j * (decoded_hd.values - true_hd_interp.values))))
print(f"Median absolute decoding error: {np.median(np.abs(error)):.1f} deg")
print(f"Circular mean resultant length of decoding error: {resultant_vector_length(np.ones(len(error)), np.radians(error)):.3f}")

Median absolute decoding error: 25.3 deg
Circular mean resultant length of decoding error: 0.756


In [16]:
# search 120s windows within the longest continuous awake bout of the test
# epoch for the one with the most head-direction movement (an objective
# criterion, chosen independent of decoding accuracy) to give a representative,
# behaviorally informative example rather than a near-stationary period
longest_bout = np.argmax(test_ep.end - test_ep.start)
bout_start, bout_end = test_ep.start[longest_bout], test_ep.end[longest_bout]

WINDOW = 120
best_range, example_start = -1, bout_start
for t0 in np.arange(bout_start, bout_end - WINDOW, 60):
    win_hd = head_direction.restrict(nap.IntervalSet(start=t0, end=t0 + WINDOW))
    if len(win_hd) < 100:
        continue
    ang_range = np.ptp(np.unwrap(win_hd.values))
    if ang_range > best_range:
        best_range, example_start = ang_range, t0

example_decode_ep = nap.IntervalSet(start=example_start, end=example_start + WINDOW)

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

true_ex = true_hd_test.restrict(example_decode_ep)
decoded_ex = decoded_hd.restrict(example_decode_ep)
axes[0].plot(true_ex.index.values, np.degrees(true_ex.values), ".", color="k", markersize=3, label="true head direction")
axes[0].plot(decoded_ex.index.values, np.degrees(decoded_ex.values), ".", color="C3", markersize=3, label="decoded (Bayesian)")
axes[0].set_ylabel("Head direction (deg)")
axes[0].set_xlabel("Time (s)")
axes[0].set_title(f"Decoded vs. true head direction, held-out test epoch (using {len(hd_unit_ids)} HD cells)")
axes[0].legend(loc="upper right", markerscale=3)

axes[1].hist(error, bins=72, range=(-180, 180), color="C0")
axes[1].set_xlabel("Decoding error (deg)")
axes[1].set_ylabel("Count")
axes[1].set_title("Distribution of decoding error across the held-out test epoch")

plt.tight_layout()
plt.savefig(f"{FIGDIR}/05_decoding.png", dpi=150)
plt.close()

## Summary

Head-direction tuning was assessed for all 22 recorded units during waking behavior in
a single session from DANDI:000056. A subset of units showed strong, statistically
significant circular tuning to head direction (shuffle test, p < 0.01), with preferred
directions tiling the full 360-degree range. Bayesian decoding using only these
significant HD cells reconstructed the animal's instantaneous heading on held-out data
well above chance, confirming that their population activity carries a coherent,
continuously updated representation of head direction — the defining property of head
direction cells.

In [17]:
io.close()
print("Done.")

Done.
